# 训练脚本

相较于普通 baseline，做了两类增强：
- **特征工程**：自动识别数值列与类别列，清理高缺失和低信息列，补充交叉特征、比值特征、短窗/长窗趋势特征，并对类别列做频次压缩与编码[cite:44]。
- **超参数优化**：使用 Optuna 分别对 LightGBM 和 XGBoost 做交叉验证搜索，主指标采用 `macro_f1`，更适合当前明显不均衡的 6 类标签分布[cite:44]。


## Cell 1 说明：安装与导入依赖

这里选择的核心建模器是 LightGBM 与 XGBoost，因为当前特征表本质上是结构化统计特征，树模型通常比纯神经网络 baseline 更容易起效，也更利于做特征重要性分析与超参数搜索。

若环境中尚未安装 `lightgbm`、`xgboost`、`optuna`，可以先取消注释安装语句。

In [1]:
# !pip install -q lightgbm xgboost optuna openpyxl

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import VotingClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import optuna

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
N_SPLITS = 5

## Cell 2 说明：读取数据并检查基础分布

这一段读取两份特征表并进行按列对齐合并，再做基础数据审查，包括数据形状、标签分布、缺失比例和列类型。当前任务标签呈明显长尾分布，这意味着训练流程必须在验证指标和样本权重上显式处理不均衡，否则模型会自然偏向头部类别[cite:44]。

你可以分别设置 `DATA_PATH_1` 和 `DATA_PATH_2`；合并时使用 `pd.concat(..., join='outer')` 自动兼容列差异，并补齐缺失值。

In [ ]:
DATA_PATH_1 = r'\\output\\features_sample_2.csv'
DATA_PATH_2 = r'\\output\\features_sample_0.csv'
DATA_PATH_3 = r'\\output\\features_sample_3.csv'
OUTPUT_DIR = Path('./output/models_023')
OUTPUT_DIR.mkdir(exist_ok=True)

df_1 = pd.read_csv(DATA_PATH_1)
df_2 = pd.read_csv(DATA_PATH_2)
df_3 = pd.read_csv(DATA_PATH_3)

# 按列对齐合并，兼容两份特征表列不完全一致
df = pd.concat([df_1, df_2, df_3], axis=0, ignore_index=True, join='outer')

print('df_1 shape:', df_1.shape)
print('df_2 shape:', df_2.shape)
print('df_3 shape:', df_3.shape)
print('merged shape:', df.shape)

print('\nlabel distribution:')
print(df['label'].value_counts(dropna=False))
print('\nhead:')
display(df.head())

missing_ratio = df.isna().mean().sort_values(ascending=False)
print('\nTop missing ratio columns:')
display(missing_ratio.head(20))

df_1 shape: (28008, 139)
df_2 shape: (96532, 139)
df_3 shape: (4768, 139)
merged shape: (129308, 139)

label distribution:
label
Fire           37826
Action         37070
SkillStart     33762
Looting        12022
Grenade         6994
BeingResuce     1634
Name: count, dtype: int64

head:


,sample_id,label,main_player_id,decision_time,last_x,last_y,last_z,last_weapon_yaw,last_weapon_pitch,last_vx,...,last_death_in_gap,nearest_player_dist,mean_player_dist,std_player_dist,recent5_path_len,recent20_path_len,recent5_disp_norm,recent20_disp_norm,path_straightness_20,decision_effect_radius
0,Action1000_2,Action,5529,20.0,6753.1,-212.6,-4543.9,25.3,317.6,0.0,...,NaN,26.702247,112.194328,83.136118,0.000000,439.426995,0.000000,0.806226,0.001835,NaN
1,Action1001_2,Action,3714,20.0,6706.1,-206.7,-4775.7,342.3,358.7,4.7,...,NaN,22.931201,209.666239,81.903236,321.207397,1242.270122,3.180271,40.801225,0.032844,NaN
2,Action1002_2,Action,3714,20.0,6701.9,-206.8,-4759.4,313.8,359.0,1.4,...,NaN,14.730241,194.921613,78.557127,178.888465,1443.555421,1.771173,37.819043,0.026199,NaN
3,Action1003_2,Action,2398,20.0,6774.1,-212.4,-4543.2,269.0,346.4,-7.3,...,NaN,9.248784,120.278871,86.937471,293.366903,697.300965,2.904623,3.813135,0.005468,NaN
4,Action1004_2,Action,2398,20.0,6755.9,-218.7,-4537.5,321.4,351.0,4.9,...,NaN,4.704253,111.891887,87.216220,411.616488,1475.473441,4.075411,26.155114,0.017727,NaN



Top missing ratio columns:


last_knock_in_gap                 1.000000
last_knock_out_gap                1.000000
last_death_in_gap                 0.999899
last_visible_state                0.978965
decision_effect_radius            0.949949
last_death_out_gap                0.935696
last_damage_in_gap                0.877146
damage_in_visible_ratio           0.877146
last_damage_out_gap               0.730125
damage_out_visible_ratio          0.730125
last_skill_hit_gap                0.472098
last_skill_cast_gap               0.442154
current_action_time               0.098153
current_action_gap_to_decision    0.098153
nearest_player_dist               0.060553
mean_player_dist                  0.060553
seg_fov_mean_diff                 0.056114
seg_scope_ratio_diff              0.056114
seg_speed_mean_diff               0.056114
seg1_scope_ratio                  0.056083
dtype: float64

## Cell 3 说明：定义原始列角色并排除明显不应直接训练的字段

这一段的目标是先把“标签列、样本标识列、可能的泄漏列”与真正可训练特征分开。像 `sample_id` 这种编号列通常只用于回查，不应直接喂模型；`main_player_id` 也更像身份标识，如果测试阶段玩家分布变化，容易导致过拟合。


In [3]:
TARGET_COL = 'label'
ID_COLS = ['sample_id']
HIGH_RISK_ID_COLS = ['main_player_id', 'decision_time']
LEAKY_COLS = ['decision_skill_name', 'decision_effect_radius']

drop_cols = [c for c in ID_COLS + HIGH_RISK_ID_COLS + LEAKY_COLS if c in df.columns]
print('drop_cols =', drop_cols)

feature_cols = [c for c in df.columns if c not in drop_cols + [TARGET_COL]]
print('feature count after initial exclusion:', len(feature_cols))

drop_cols = ['sample_id', 'main_player_id', 'decision_time', 'decision_skill_name', 'decision_effect_radius']
feature count after initial exclusion: 133


## Cell 4 说明：基础特征工程函数

这一段是本 Notebook 的核心增强之一：在原始统计特征基础上继续构造一批更适合树模型学习的派生特征。当前原表已经包含多时间窗统计（20/10/5/3 秒）、动作/伤害/技能/路径特征。

主要增强思路包括：
- **短窗 / 长窗比值**：例如 `recent3_speed_mean / recent20_speed_mean`，用于描述决策前状态是否突然变化。
- **战斗净值与强度差**：例如近期输出伤害与输入伤害的差值、近 3 秒与近 20 秒战斗密度比。
- **空间与行为耦合**：例如路径长度与直线位移的耦合、最近玩家距离与开镜比例的联合表达。
- **类别列清洗**：将过稀有类别合并为 `__OTHER__`，降低 One-Hot 维度碎片化。


In [4]:
def safe_div(a, b):
    return a / (b.replace(0, np.nan) if isinstance(b, pd.Series) else (np.nan if b == 0 else b))

def rare_category_bucket(series, min_count=30):
    vc = series.astype(str).value_counts(dropna=False)
    keep = set(vc[vc >= min_count].index)
    return series.astype(str).apply(lambda x: x if x in keep else '__OTHER__')

def add_engineered_features(df_in):
    df = df_in.copy()

    # 速度/视角/开镜趋势比值
    if {'recent3_speed_mean', 'recent20_speed_mean'}.issubset(df.columns):
        df['speed_ratio_3_20'] = safe_div(df['recent3_speed_mean'], df['recent20_speed_mean'] + 1e-6)
    if {'recent5_speed_mean', 'recent20_speed_mean'}.issubset(df.columns):
        df['speed_ratio_5_20'] = safe_div(df['recent5_speed_mean'], df['recent20_speed_mean'] + 1e-6)
    if {'recent3_scope_ratio', 'recent20_scope_ratio'}.issubset(df.columns):
        df['scope_ratio_3_20'] = safe_div(df['recent3_scope_ratio'] + 1e-6, df['recent20_scope_ratio'] + 1e-6)
    if {'recent5_scope_ratio', 'recent20_scope_ratio'}.issubset(df.columns):
        df['scope_ratio_5_20'] = safe_div(df['recent5_scope_ratio'] + 1e-6, df['recent20_scope_ratio'] + 1e-6)
    if {'recent3_fov_mean', 'recent20_fov_mean'}.issubset(df.columns):
        df['fov_ratio_3_20'] = safe_div(df['recent3_fov_mean'] + 1e-6, df['recent20_fov_mean'] + 1e-6)

    # 战斗强度与净值
    if {'recent3_damage_out_count', 'recent3_damage_in_count'}.issubset(df.columns):
        df['recent3_damage_count_net'] = df['recent3_damage_out_count'] - df['recent3_damage_in_count']
    if {'recent5_damage_out_count', 'recent5_damage_in_count'}.issubset(df.columns):
        df['recent5_damage_count_net'] = df['recent5_damage_out_count'] - df['recent5_damage_in_count']
    if {'recent20_damage_out_hp_sum', 'recent20_damage_in_hp_sum'}.issubset(df.columns):
        df['recent20_damage_hp_net_v2'] = df['recent20_damage_out_hp_sum'] - df['recent20_damage_in_hp_sum']
    if {'recent5_damage_out_hp_sum', 'recent5_damage_in_hp_sum'}.issubset(df.columns):
        df['recent5_damage_hp_net_v2'] = df['recent5_damage_out_hp_sum'] - df['recent5_damage_in_hp_sum']
    if {'recent3_damage_out_count', 'recent20_damage_out_count'}.issubset(df.columns):
        df['damage_out_ratio_3_20'] = safe_div(df['recent3_damage_out_count'] + 1e-6, df['recent20_damage_out_count'] + 1e-6)
    if {'recent3_damage_in_count', 'recent20_damage_in_count'}.issubset(df.columns):
        df['damage_in_ratio_3_20'] = safe_div(df['recent3_damage_in_count'] + 1e-6, df['recent20_damage_in_count'] + 1e-6)

    # 技能与动作密度
    if {'recent5_skill_cast_count', 'recent20_skill_cast_count'}.issubset(df.columns):
        df['skill_cast_ratio_5_20'] = safe_div(df['recent5_skill_cast_count'] + 1e-6, df['recent20_skill_cast_count'] + 1e-6)
    if {'recent5_action_count', 'recent20_frames'}.issubset(df.columns):
        df['action_density_5'] = safe_div(df['recent5_action_count'], df['recent20_frames'] + 1e-6)
    if {'recent3_action_count', 'recent20_frames'}.issubset(df.columns):
        df['action_density_3'] = safe_div(df['recent3_action_count'], df['recent20_frames'] + 1e-6)

    # 路径与空间耦合
    if {'recent20_path_len', 'recent20_disp_norm'}.issubset(df.columns):
        df['path_efficiency_20'] = safe_div(df['recent20_disp_norm'] + 1e-6, df['recent20_path_len'] + 1e-6)
    if {'recent5_path_len', 'recent5_disp_norm'}.issubset(df.columns):
        df['path_efficiency_5'] = safe_div(df['recent5_disp_norm'] + 1e-6, df['recent5_path_len'] + 1e-6)
    if {'nearest_player_dist', 'mean_player_dist'}.issubset(df.columns):
        df['nearest_mean_dist_ratio'] = safe_div(df['nearest_player_dist'] + 1e-6, df['mean_player_dist'] + 1e-6)

    # 行为趋势差
    if {'seg1_speed_mean', 'seg2_speed_mean'}.issubset(df.columns):
        df['seg_speed_mean_absdiff'] = (df['seg2_speed_mean'] - df['seg1_speed_mean']).abs()
    if {'seg1_scope_ratio', 'seg2_scope_ratio'}.issubset(df.columns):
        df['seg_scope_ratio_absdiff'] = (df['seg2_scope_ratio'] - df['seg1_scope_ratio']).abs()
    if {'seg1_fov_mean', 'seg2_fov_mean'}.issubset(df.columns):
        df['seg_fov_mean_absdiff'] = (df['seg2_fov_mean'] - df['seg1_fov_mean']).abs()

    # 决策前临战标记
    if 'recent3_damage_in_count' in df.columns:
        df['flag_recent3_under_attack'] = (df['recent3_damage_in_count'] > 0).astype(int)
    if 'recent3_damage_out_count' in df.columns:
        df['flag_recent3_attack_out'] = (df['recent3_damage_out_count'] > 0).astype(int)
    if 'recent3_scope_ratio' in df.columns:
        df['flag_recent3_scoped'] = (df['recent3_scope_ratio'] > 0).astype(int)
    if 'recent20_skill_cast_count' in df.columns:
        df['flag_recent20_skill_used'] = (df['recent20_skill_cast_count'] > 0).astype(int)

    # 清理类别列：稀有类别压缩
    cat_candidates = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    # 注意不要对 TARGET_COL 做清洗
    cat_candidates = [c for c in cat_candidates if c != TARGET_COL]
    
    for c in cat_candidates:
        df[c] = rare_category_bucket(df[c], min_count=30)

    return df

df_fe = add_engineered_features(df[feature_cols + [TARGET_COL]].copy())
print('shape after feature engineering:', df_fe.shape)

shape after feature engineering: (129308, 158)


## Cell 5 说明：删除高缺失、低信息特征

这一段进一步做特征清洗。原因是树模型虽然对缺失值相对友好，但大量高缺失或几乎恒定的列会增加搜索噪声、拖慢调参，并可能让模型学到偶然模式。
- 缺失率超过阈值的列直接删除。
- 去掉唯一值过少（例如全表几乎只有一个取值）的低信息列。


In [5]:
MAX_MISSING_RATIO = 0.99 # 放宽缺失率阈值（如92%的last_damage_in_gap代表大多数时候没受伤，这本身是有效特征）
MIN_UNIQUE = 2

feature_only_cols = [c for c in df_fe.columns if c != TARGET_COL]
miss_ratio = df_fe[feature_only_cols].isna().mean()
high_missing_cols = miss_ratio[miss_ratio > MAX_MISSING_RATIO].index.tolist()

nunique_series = df_fe[feature_only_cols].nunique(dropna=False)
low_info_cols = nunique_series[nunique_series < MIN_UNIQUE].index.tolist()

final_drop_cols = sorted(set(high_missing_cols + low_info_cols))
df_model = df_fe.drop(columns=final_drop_cols)

print('dropped high-missing / low-info cols:', len(final_drop_cols))
print('final shape:', df_model.shape)
display(pd.DataFrame({'col': final_drop_cols}).head(50))

dropped high-missing / low-info cols: 3
final shape: (129308, 155)


,col
0,last_death_in_gap
1,last_knock_in_gap
2,last_knock_out_gap


## Cell 6 说明：准备训练数据与预处理管道

把特征分成数值列和类别列，并建立统一的预处理管道。由于当前特征表中既有大量数值统计列，也有 `action_mode_20`、`current_action_name`、`prev_action_name` 等类别列，因此需要分别处理。

数值列采用中位数填补，类别列采用众数填补 + One-Hot 编码，这种方案简单稳健，适合树模型和交叉验证搜索。标签则使用 `LabelEncoder` 转成整数类标。

In [6]:
X = df_model.drop(columns=[TARGET_COL]).copy()
y_raw = df_model[TARGET_COL].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = list(label_encoder.classes_)
print('classes:', class_names)

cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print('num_cols:', len(num_cols), 'cat_cols:', len(cat_cols))

# 对于数值列，高缺失（如无伤害差值）被填为-999，这种常量更容易被树模型识别出“事件未发生”
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=-999))]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10))
        ]), cat_cols),
    ],
    remainder='drop'
)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

classes: ['Action', 'BeingResuce', 'Fire', 'Grenade', 'Looting', 'SkillStart']
num_cols: 149 cat_cols: 5


## Cell 7 说明：先建立一个稳健 baseline

先跑一个相对稳健的 baseline 。这样可以知道：当前特征工程有没有带来提升、是否出现异常欠拟合或过拟合，以及后面调参的收益空间大概有多大。

先使用一个较保守的 LightGBM baseline，并用 5 折 `macro_f1` 做评估。对于当前这种不均衡多分类任务，`macro_f1` 比单纯 accuracy 更能反映模型是否照顾到 `Grenade` 和 `BeingResuce` 这类长尾类别。

In [7]:
baseline_model = LGBMClassifier(
    objective='multiclass',
    num_class=len(class_names),
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=RANDOM_STATE,
    class_weight='balanced',
    verbose=-1
)

baseline_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', baseline_model)
])

baseline_scores = cross_val_score(baseline_pipe, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
print('Baseline macro_f1 scores:', baseline_scores)
print('Baseline macro_f1 mean:', baseline_scores.mean())

Baseline macro_f1 scores: [0.73964631 0.73699898 0.7370733  0.73749535 0.73747801]
Baseline macro_f1 mean: 0.7377383901396004


## 开关单元：快速实验控制

运行这个单元后，可以通过布尔开关快速切换流程：
- 是否执行 LightGBM / XGBoost 的 Optuna 超参优化
- 是否启用增强样本权重（提升小类关注）
- 是否启用 LightGBM + XGBoost 概率融合（Soft Voting）

当超参优化开关关闭时，后续对应单元会自动跳过优化，并在 Cell 12 使用默认参数继续训练与对比。

In [ ]:
# 快速实验开关（只需改这里）
ENABLE_OPTUNA_LGB = True
ENABLE_OPTUNA_XGB = False

LGB_TRIALS = 30
XGB_TRIALS = 50

# 增强训练开关
ENABLE_ENHANCED_SAMPLE_WEIGHT = True
RARE_CLASS_WEIGHT_MULTIPLIER = {
    'BeingResuce': 1.8,
    'Grenade': 2.4,
    'SkillStart': 1.2
}

# 业务导向惩罚：提升“交战/避战”组别学习强度
ENABLE_INTENT_GROUP_WEIGHT = True
INTENT_GROUP_MULTIPLIER = {
    '交战': 1.2,
    '避战': 1.6
}

ENABLE_SOFT_VOTING = True
SOFT_VOTING_WEIGHTS = (0.45, 0.55)  # (LightGBM, XGBoost)

print('Switches:')
print('  ENABLE_OPTUNA_LGB =', ENABLE_OPTUNA_LGB)
print('  ENABLE_OPTUNA_XGB =', ENABLE_OPTUNA_XGB)
print('  LGB_TRIALS =', LGB_TRIALS)
print('  XGB_TRIALS =', XGB_TRIALS)
print('  ENABLE_ENHANCED_SAMPLE_WEIGHT =', ENABLE_ENHANCED_SAMPLE_WEIGHT)
print('  RARE_CLASS_WEIGHT_MULTIPLIER =', RARE_CLASS_WEIGHT_MULTIPLIER)
print('  ENABLE_INTENT_GROUP_WEIGHT =', ENABLE_INTENT_GROUP_WEIGHT)
print('  INTENT_GROUP_MULTIPLIER =', INTENT_GROUP_MULTIPLIER)
print('  ENABLE_SOFT_VOTING =', ENABLE_SOFT_VOTING)
print('  SOFT_VOTING_WEIGHTS =', SOFT_VOTING_WEIGHTS)

Switches:
  ENABLE_OPTUNA_LGB = False
  ENABLE_OPTUNA_XGB = True
  LGB_TRIALS = 20
  XGB_TRIALS = 50
  ENABLE_ENHANCED_SAMPLE_WEIGHT = True
  RARE_CLASS_WEIGHT_MULTIPLIER = {'BeingResuce': 1.8, 'Grenade': 2.4, 'SkillStart': 1.2}
  ENABLE_INTENT_GROUP_WEIGHT = True
  INTENT_GROUP_MULTIPLIER = {'交战': 1.2, '避战': 1.8}
  ENABLE_SOFT_VOTING = True
  SOFT_VOTING_WEIGHTS = (0.45, 0.55)


## Cell 8 说明：定义 Optuna 搜索目标（LightGBM）

这一段正式进入超参数优化。LightGBM 对结构化特征通常表现不错，但它也容易因为 `num_leaves`、`min_child_samples`、`subsample` 等参数设置不当而过拟合。

这里用 Optuna 搜索一组更适合当前任务的数据驱动参数，并且每次 trial 都用 5 折 `macro_f1` 做目标函数。这样得到的最优参数通常比只在单次切分上调参更稳健。

In [9]:
def objective_lgb(trial):
    params = {
        'objective': 'multiclass',
        'num_class': len(class_names),
        'n_estimators': trial.suggest_int('n_estimators', 200, 900),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.65, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'random_state': RANDOM_STATE,
        'class_weight': 'balanced',
        'verbose': -1
    }

    model = LGBMClassifier(**params)
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])

    scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

## Cell 9 说明：执行 LightGBM 超参数搜索

In [10]:
if ENABLE_OPTUNA_LGB:
    study_lgb = optuna.create_study(direction='maximize', study_name='lgb_macro_f1')
    study_lgb.optimize(objective_lgb, n_trials=LGB_TRIALS, show_progress_bar=True)

    print('Best LGB score:', study_lgb.best_value)
    print('Best LGB params:')
    print(json.dumps(study_lgb.best_params, indent=2, ensure_ascii=False))
else:
    study_lgb = None
    print('Skip LightGBM Optuna (ENABLE_OPTUNA_LGB=False). Cell 12 will use default parameters.')

Skip LightGBM Optuna (ENABLE_OPTUNA_LGB=False). Cell 12 will use default parameters.


## Cell 10 说明：定义 Optuna 搜索目标（XGBoost）


同样用 5 折 `macro_f1` 搜索 XGBoost 参数，重点搜索 `max_depth`、`min_child_weight`、`subsample`、`colsample_bytree`、正则项等参数，因为它们对控制过拟合和长尾类别泛化非常关键。

In [11]:
def objective_xgb(trial):
    params = {
        'objective': 'multi:softprob',
        'num_class': len(class_names),
        'n_estimators': trial.suggest_int('n_estimators', 200, 900),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.12, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.65, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-4, 3.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'tree_method': 'hist',
        'random_state': RANDOM_STATE,
        'eval_metric': 'mlogloss',
        'n_jobs': -1,
        'verbosity': 0
    }

    model = XGBClassifier(**params)
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', model)
    ])

    scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

## Cell 11 说明：执行 XGBoost 超参数搜索

与 LightGBM 搜索相同，最终会给出一组最优 XGBoost 参数。后面可以比较两者谁在交叉验证上更好，也可以考虑把两者做简单概率融合。


In [12]:
if ENABLE_OPTUNA_XGB:
    study_xgb = optuna.create_study(direction='maximize', study_name='xgb_macro_f1')
    study_xgb.optimize(objective_xgb, n_trials=XGB_TRIALS, show_progress_bar=True)

    print('Best XGB score:', study_xgb.best_value)
    print('Best XGB params:')
    print(json.dumps(study_xgb.best_params, indent=2, ensure_ascii=False))
else: 
    study_xgb = None
    print('Skip XGBoost Optuna (ENABLE_OPTUNA_XGB=False). Cell 12 will use default parameters.')

[I 2026-04-11 18:46:16,121] A new study created in memory with name: xgb_macro_f1


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-11 18:49:29,887] Trial 0 finished with value: 0.8321862644995612 and parameters: {'n_estimators': 412, 'learning_rate': 0.02253276429676527, 'max_depth': 9, 'min_child_weight': 3, 'subsample': 0.7637203675793444, 'colsample_bytree': 0.6663638737707109, 'gamma': 0.00012989483852293577, 'reg_alpha': 0.014134950045933254, 'reg_lambda': 0.00011222619854598657}. Best is trial 0 with value: 0.8321862644995612.
[I 2026-04-11 18:55:40,746] Trial 1 finished with value: 0.8780827502324223 and parameters: {'n_estimators': 812, 'learning_rate': 0.11945572171819599, 'max_depth': 8, 'min_child_weight': 3, 'subsample': 0.956882748553209, 'colsample_bytree': 0.9763358894563274, 'gamma': 0.0018149515375244295, 'reg_alpha': 0.5408212206812245, 'reg_lambda': 1.6930599734486114}. Best is trial 1 with value: 0.8780827502324223.
[I 2026-04-11 18:58:58,884] Trial 2 finished with value: 0.8196291612216948 and parameters: {'n_estimators': 584, 'learning_rate': 0.020506402398075444, 'max_depth': 7, '

## Cell 12 说明：使用最优参数重新做交叉验证对比

有了最优参数后，需要重新在统一验证框架下比较 baseline、最优 LightGBM、最优 XGBoost 的表现，确认调参是否真正带来了提升，而不是单次 trial 的偶然波动。

In [13]:
import numpy as np
from optuna.trial import TrialState


def has_completed_trials(study_obj):
    try:
        if study_obj is None:
            return False
        completed = [t for t in study_obj.trials if t.state == TrialState.COMPLETE]
        return len(completed) > 0
    except Exception:
        return False


# 若未执行超参优化，或没有完成的 trial，回退到稳健默认参数
study_lgb_obj = globals().get('study_lgb', None)
study_xgb_obj = globals().get('study_xgb', None)

use_lgb_optuna = has_completed_trials(study_lgb_obj)
use_xgb_optuna = has_completed_trials(study_xgb_obj)

if use_lgb_optuna:
    lgb_kwargs = dict(study_lgb_obj.best_params)
    print('Cell 12: LightGBM 使用 Optuna 最优参数。')
else:
    lgb_kwargs = {
        'n_estimators': 400,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': -1,
        'min_child_samples': 20,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.0,
        'reg_lambda': 0.0
    }
    print('Cell 12: 未检测到 LightGBM 已完成的超参优化 trial，使用默认参数。')

if use_xgb_optuna:
    xgb_kwargs = dict(study_xgb_obj.best_params)
    print('Cell 12: XGBoost 使用 Optuna 最优参数。')
else:
    xgb_kwargs = {
        'n_estimators': 400,
        'learning_rate': 0.05,
        'max_depth': 6,
        'min_child_weight': 1,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'gamma': 0.0,
        'reg_alpha': 0.0,
        'reg_lambda': 1.0
    }
    print('Cell 12: 未检测到 XGBoost 已完成的超参优化 trial，使用默认参数。')

best_lgb = LGBMClassifier(
    objective='multiclass',
    num_class=len(class_names),
    random_state=RANDOM_STATE,
    class_weight='balanced',
    verbose=-1,
    **lgb_kwargs
)

best_xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=len(class_names),
    tree_method='hist',
    random_state=RANDOM_STATE,
    eval_metric='mlogloss',
    n_jobs=-1,
    verbosity=0,
    **xgb_kwargs
)

best_lgb_pipe = Pipeline([('preprocess', preprocessor), ('model', best_lgb)])
best_xgb_pipe = Pipeline([('preprocess', preprocessor), ('model', best_xgb)])

best_lgb_scores = cross_val_score(best_lgb_pipe, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
best_xgb_scores = cross_val_score(best_xgb_pipe, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)

# 同时统计验证集 accuracy，便于和 macro_f1 一起观察
best_lgb_acc_scores = cross_val_score(best_lgb_pipe, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
best_xgb_acc_scores = cross_val_score(best_xgb_pipe, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

cv_macro_scores = {
    'LightGBM': float(best_lgb_scores.mean()),
    'XGBoost': float(best_xgb_scores.mean())
}

print('=' * 50)
print('          CROSS VALIDATION RESULTS (5 Fold)')
print('=' * 50)
print(f'LightGBM F1-Macro CV Scores : {np.round(best_lgb_scores, 4)}')
print(f'LightGBM F1-Macro MEAN      : {best_lgb_scores.mean():.4f} +/- {best_lgb_scores.std():.4f}')
print(f'LightGBM Accuracy CV Scores : {np.round(best_lgb_acc_scores, 4)}')
print(f'LightGBM Accuracy MEAN      : {best_lgb_acc_scores.mean():.4f} +/- {best_lgb_acc_scores.std():.4f}')
print('-' * 50)
print(f'XGBoost  F1-Macro CV Scores : {np.round(best_xgb_scores, 4)}')
print(f'XGBoost  F1-Macro MEAN      : {best_xgb_scores.mean():.4f} +/- {best_xgb_scores.std():.4f}')
print(f'XGBoost  Accuracy CV Scores : {np.round(best_xgb_acc_scores, 4)}')
print(f'XGBoost  Accuracy MEAN      : {best_xgb_acc_scores.mean():.4f} +/- {best_xgb_acc_scores.std():.4f}')

if ENABLE_SOFT_VOTING:
    ensemble_model = VotingClassifier(
        estimators=[('lgb', best_lgb), ('xgb', best_xgb)],
        voting='soft',
        weights=list(SOFT_VOTING_WEIGHTS)
    )
    best_ensemble_pipe = Pipeline([('preprocess', preprocessor), ('model', ensemble_model)])
    best_ensemble_scores = cross_val_score(best_ensemble_pipe, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
    best_ensemble_acc_scores = cross_val_score(best_ensemble_pipe, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

    cv_macro_scores['SoftVoting'] = float(best_ensemble_scores.mean())

    print('-' * 50)
    print(f'SoftVoting F1-Macro CV Scores: {np.round(best_ensemble_scores, 4)}')
    print(f'SoftVoting F1-Macro MEAN     : {best_ensemble_scores.mean():.4f} +/- {best_ensemble_scores.std():.4f}')
    print(f'SoftVoting Accuracy Scores   : {np.round(best_ensemble_acc_scores, 4)}')
    print(f'SoftVoting Accuracy MEAN     : {best_ensemble_acc_scores.mean():.4f} +/- {best_ensemble_acc_scores.std():.4f}')

print('=' * 50)
selected_cv_model_name = max(cv_macro_scores, key=cv_macro_scores.get)
print(f'==> Selected by CV macro_f1: {selected_cv_model_name} ({cv_macro_scores[selected_cv_model_name]:.4f})')

Cell 12: 未检测到 LightGBM 已完成的超参优化 trial，使用默认参数。
Cell 12: XGBoost 使用 Optuna 最优参数。
          CROSS VALIDATION RESULTS (5 Fold)
LightGBM F1-Macro CV Scores : [0.7396 0.737  0.7371 0.7375 0.7375]
LightGBM F1-Macro MEAN      : 0.7377 +/- 0.0010
LightGBM Accuracy CV Scores : [0.8185 0.8143 0.8153 0.8152 0.8174]
LightGBM Accuracy MEAN      : 0.8161 +/- 0.0016
--------------------------------------------------
XGBoost  F1-Macro CV Scores : [0.8836 0.8801 0.8745 0.8783 0.8757]
XGBoost  F1-Macro MEAN      : 0.8784 +/- 0.0032
XGBoost  Accuracy CV Scores : [0.8942 0.891  0.8882 0.8905 0.8915]
XGBoost  Accuracy MEAN      : 0.8911 +/- 0.0019
--------------------------------------------------
SoftVoting F1-Macro CV Scores: [0.8814 0.878  0.8746 0.8771 0.8788]
SoftVoting F1-Macro MEAN     : 0.8780 +/- 0.0022
SoftVoting Accuracy Scores   : [0.8901 0.8865 0.8847 0.8868 0.8905]
SoftVoting Accuracy MEAN     : 0.8877 +/- 0.0022
==> Selected by CV macro_f1: XGBoost (0.8784)


## Cell 13 说明：训练最终模型并输出训练集内诊断结果

这一段在全量训练集上拟合最终模型，并输出训练集内的分类报告与混淆矩阵。这里的训练集结果不能视为真实泛化成绩，但它非常适合做错误类型诊断，例如：
- 哪些类别最容易混淆；
- `Grenade`、`BeingResuce` 是否完全学不到；
- 某些类别是否被头部类吞没。

In [14]:
import os
os.environ["LGBM_COMPAT_WARNING"] = "0"
import warnings
warnings.filterwarnings("ignore")


def build_train_sample_weight(y_labels):
    base_w = compute_sample_weight(class_weight='balanced', y=y_labels)
    w = base_w.astype(float).copy()

    if ENABLE_ENHANCED_SAMPLE_WEIGHT:
        y_str = pd.Series(y_labels).astype(str)
        for cls_name, mult in RARE_CLASS_WEIGHT_MULTIPLIER.items():
            w[y_str == cls_name] *= float(mult)

    # 业务重点：增强“交战/避战”组别惩罚
    if ENABLE_INTENT_GROUP_WEIGHT:
        y_str = pd.Series(y_labels).astype(str)
        avoid_labels = {'BeingResuce', 'Looting'}
        is_avoid = y_str.isin(avoid_labels)
        w[is_avoid.values] *= float(INTENT_GROUP_MULTIPLIER.get('避战', 1.0))
        w[(~is_avoid).values] *= float(INTENT_GROUP_MULTIPLIER.get('交战', 1.0))

    return w


train_sample_weight = build_train_sample_weight(y_raw)

# ========== 1. 拟合并评估 LightGBM ==========
print('=== Evaluating Best LightGBM on Full Train Set ===')
best_lgb_pipe.named_steps['model'].set_params(verbose=-1)
lgb_fit_kwargs = {'model__sample_weight': train_sample_weight}
best_lgb_pipe.fit(X, y, **lgb_fit_kwargs)
lgb_train_pred = best_lgb_pipe.predict(X)

print('LightGBM Train accuracy:', accuracy_score(y, lgb_train_pred))
print('LightGBM Train macro_f1:', f1_score(y, lgb_train_pred, average='macro'))
print('\nLightGBM Classification report:')
print(classification_report(y, lgb_train_pred, target_names=class_names, digits=4))

# ========== 2. 拟合并评估 XGBoost ==========
print('-' * 60)
print('=== Evaluating Best XGBoost on Full Train Set ===')
best_xgb_pipe.named_steps['model'].set_params(verbosity=0)
xgb_fit_kwargs = {'model__sample_weight': train_sample_weight}
best_xgb_pipe.fit(X, y, **xgb_fit_kwargs)
xgb_train_pred = best_xgb_pipe.predict(X)

print('XGBoost Train accuracy:', accuracy_score(y, xgb_train_pred))
print('XGBoost Train macro_f1:', f1_score(y, xgb_train_pred, average='macro'))
print('\nXGBoost Classification report:')
print(classification_report(y, xgb_train_pred, target_names=class_names, digits=4))

# ========== 3. 可选：概率融合模型 ==========
ensemble_pipe = None
ensemble_train_pred = None
if ENABLE_SOFT_VOTING:
    print('-' * 60)
    print('=== Evaluating Soft Voting (LGB + XGB) on Full Train Set ===')
    ensemble_model = VotingClassifier(
        estimators=[('lgb', best_lgb), ('xgb', best_xgb)],
        voting='soft',
        weights=list(SOFT_VOTING_WEIGHTS)
    )
    ensemble_pipe = Pipeline([('preprocess', preprocessor), ('model', ensemble_model)])
    ensemble_fit_kwargs = {'model__sample_weight': train_sample_weight}
    ensemble_pipe.fit(X, y, **ensemble_fit_kwargs)
    ensemble_train_pred = ensemble_pipe.predict(X)

    print('SoftVoting Train accuracy:', accuracy_score(y, ensemble_train_pred))
    print('SoftVoting Train macro_f1:', f1_score(y, ensemble_train_pred, average='macro'))
    print('\nSoftVoting Classification report:')
    print(classification_report(y, ensemble_train_pred, target_names=class_names, digits=4))

# ========== 4. 选定最终模型（优先使用 Cell 12 的CV选择结果） ==========
print('-' * 60)
selected_name = globals().get('selected_cv_model_name', None)
if selected_name not in {'LightGBM', 'XGBoost', 'SoftVoting'}:
    selected_name = 'LightGBM' if best_lgb_scores.mean() >= best_xgb_scores.mean() else 'XGBoost'

if selected_name == 'SoftVoting' and ensemble_pipe is None:
    print('CV selected SoftVoting, but ENABLE_SOFT_VOTING=False. Fallback to best single model by macro_f1.')
    selected_name = 'LightGBM' if best_lgb_scores.mean() >= best_xgb_scores.mean() else 'XGBoost'

if selected_name == 'LightGBM':
    final_pipe = best_lgb_pipe
    final_train_pred = lgb_train_pred
elif selected_name == 'XGBoost':
    final_pipe = best_xgb_pipe
    final_train_pred = xgb_train_pred
else:
    final_pipe = ensemble_pipe
    final_train_pred = ensemble_train_pred

final_model_name = selected_name

print(f'=> Selected final model for export: {final_model_name}')
print('\nFinal Model Confusion Matrix (on train set):')
cm = confusion_matrix(y, final_train_pred)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
display(cm_df)

=== Evaluating Best LightGBM on Full Train Set ===


Exception in thread Thread-7 (_readerthread):
Traceback (most recent call last):
  File "d:\vary\Miniforge\envs\dp\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "d:\vary\Miniforge\envs\dp\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "d:\vary\Miniforge\envs\dp\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xce in position 4: invalid continuation byte


LightGBM Train accuracy: 0.7905698023324156
LightGBM Train macro_f1: 0.7138280811143334

LightGBM Classification report:
              precision    recall  f1-score   support

      Action     0.9441    0.7884    0.8593     37070
 BeingResuce     0.1907    1.0000    0.3203      1634
        Fire     0.9674    0.7838    0.8660     37826
     Grenade     0.3172    0.9254    0.4725      6994
     Looting     0.9919    0.9987    0.9953     12022
  SkillStart     0.8727    0.6883    0.7696     33762

    accuracy                         0.7906    129308
   macro avg     0.7140    0.8641    0.7138    129308
weighted avg     0.8933    0.7906    0.8227    129308

------------------------------------------------------------
=== Evaluating Best XGBoost on Full Train Set ===
XGBoost Train accuracy: 0.9458888854517895
XGBoost Train macro_f1: 0.8638436119005003

XGBoost Classification report:
              precision    recall  f1-score   support

      Action     1.0000    0.9295    0.9634     3707

,Action,BeingResuce,Fire,Grenade,Looting,SkillStart
Action,34455,2615,0,0,0,0
BeingResuce,0,1634,0,0,0,0
Fire,0,1047,36741,33,0,5
Grenade,0,522,0,6471,1,0
Looting,0,12,0,0,12010,0
SkillStart,0,2738,0,24,0,31000


## Cell 14 说明：保存最终模型、标签映射、最优参数与特征列清单

- 被删掉的列；
- 特征工程规则；
- 标签编码器；
- 最优参数；
- 最终模型。

In [15]:
import joblib


def get_study_snapshot(study_obj):
    try:
        return {
            'has_completed_trial': True,
            'best_params': dict(study_obj.best_params),
            'best_score': float(study_obj.best_value)
        }
    except Exception:
        return {
            'has_completed_trial': False,
            'best_params': None,
            'best_score': None
        }


lgb_snapshot = get_study_snapshot(globals().get('study_lgb', None))
xgb_snapshot = get_study_snapshot(globals().get('study_xgb', None))

joblib.dump(final_pipe, OUTPUT_DIR / 'final_pipeline.joblib')
joblib.dump(label_encoder, OUTPUT_DIR / 'label_encoder.joblib')

meta = {
    'selected_model': final_model_name,
    'drop_cols': drop_cols,
    'final_drop_cols': final_drop_cols,
    'feature_columns_before_preprocess': list(X.columns),
    'class_names': class_names,
    'best_lgb_params': lgb_snapshot['best_params'],
    'best_lgb_score': lgb_snapshot['best_score'],
    'best_xgb_params': xgb_snapshot['best_params'],
    'best_xgb_score': xgb_snapshot['best_score'],
    'lgb_optuna_completed': lgb_snapshot['has_completed_trial'],
    'xgb_optuna_completed': xgb_snapshot['has_completed_trial']
}

with open(OUTPUT_DIR / 'training_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('Saved to:', OUTPUT_DIR.resolve())
print(list(OUTPUT_DIR.iterdir()))

Saved to: E:\EOne\2026游戏安全技术竞赛-游戏安全AI方向-初赛\output\models_023
[WindowsPath('output/models_023/final_pipeline.joblib'), WindowsPath('output/models_023/label_encoder.joblib'), WindowsPath('output/models_023/training_meta.json')]
